# Download MEVID locally and upload it to Kaggle

Run this notebook with a **local Windows Python kernel**. It downloads and extracts MEVID into `D:\Study\ImageProcess\MultiCameraTracking\data\mevid`, validates the extracted folders, and optionally uploads that directory to `qimatx/mevid` with KaggleHub.

The archives are large. Completed downloads and extractions are reused on reruns. This notebook does not use `/kaggle/input` or `/kaggle/working`.

In [ ]:
from pathlib import Path
import os
import shutil
import tarfile
import time
import urllib.request
import zipfile

PROJECT_ROOT = Path(r"D:\Study\ImageProcess\MultiCameraTracking")
DATA_DIR = PROJECT_ROOT / "data"
MEVID_LOCAL_DIR = DATA_DIR / "mevid"

# Set this to False when you only want to download/validate locally.
UPLOAD_TO_KAGGLE = True
KAGGLE_DATASET_HANDLE = "qimatx/mevid"

DATA_DIR.mkdir(parents=True, exist_ok=True)
MEVID_LOCAL_DIR.mkdir(parents=True, exist_ok=True)

print(f"Local MEVID directory: {MEVID_LOCAL_DIR}")
print(f"Free disk space: {shutil.disk_usage(DATA_DIR).free / 1024**3:.1f} GiB")

In [ ]:
BASE_URL = "https://mevadata-public-01.s3.amazonaws.com/mevid-annotations"


def download(url: str, destination: Path) -> None:
    """Download to a .part file and resume it when the server supports ranges."""
    partial = destination.with_name(destination.name + ".part")
    offset = partial.stat().st_size if partial.exists() else 0
    headers = {"Range": f"bytes={offset}-"} if offset else {}
    request = urllib.request.Request(url, headers=headers)

    with urllib.request.urlopen(request) as response:
        mode = "ab" if offset and response.status == 206 else "wb"
        downloaded = offset if mode == "ab" else 0
        length = int(response.headers.get("Content-Length", 0))
        total = downloaded + length if length else 0
        started = time.monotonic()
        start_bytes = downloaded
        next_percent = (downloaded * 10 // total + 1) * 10 if total else None

        if downloaded:
            print(f"Resuming from {downloaded / 1024**3:.2f} GiB", flush=True)

        with partial.open(mode) as output:
            while chunk := response.read(1024 * 1024):
                output.write(chunk)
                downloaded += len(chunk)
                now = time.monotonic()
                if total and downloaded * 100 >= next_percent * total:
                    speed = (downloaded - start_bytes) / max(now - started, 0.001)
                    eta = f", ETA {(total - downloaded) / speed / 60:.1f} min" if speed else ""
                    print(
                        f"{destination.name}: {next_percent}% "
                        f"({downloaded / 1024**3:.2f} GiB), "
                        f"{speed / 1024**2:.1f} MiB/s{eta}",
                        flush=True,
                    )
                    next_percent += 10

        if total and downloaded != total:
            raise IOError(
                f"Incomplete download: {downloaded} of {total} bytes; rerun to resume"
            )

    partial.replace(destination)


def extract_archive(archive: Path, destination: Path) -> None:
    """Extract an official MEVID archive while rejecting unsafe paths."""
    destination = destination.resolve()
    if archive.suffix == ".zip":
        with zipfile.ZipFile(archive) as source:
            for member in source.infolist():
                target = (destination / member.filename).resolve()
                if target != destination and destination not in target.parents:
                    raise RuntimeError(f"Unsafe ZIP member: {member.filename}")
            source.extractall(destination)
    else:
        with tarfile.open(archive, "r:gz") as source:
            source.extractall(destination, filter="data")


ARCHIVES = [
    ("Annotations", "mevid-v1-annotation-data.zip", "mevid-v1-annotation-data"),
    ("Train images", "mevid-v1-bbox-train.tgz", "bbox_train"),
    ("Test images", "mevid-v1-bbox-test.tgz", "bbox_test"),
]

for label, filename, expected_folder in ARCHIVES:
    archive = DATA_DIR / filename
    marker = MEVID_LOCAL_DIR / f".{filename}.extracted"
    legacy_marker = DATA_DIR / f".{filename}.extracted"
    expected_path = MEVID_LOCAL_DIR / expected_folder

    # Older notebook versions wrote markers beside data/mevid. Honour them so the
    # already extracted 43+ GiB dataset is not downloaded again.
    if expected_path.is_dir() and (marker.exists() or legacy_marker.exists()):
        marker.touch(exist_ok=True)
        print(f"{label}: already extracted; skipping")
        continue

    print(f"{label}: preparing {filename}...", flush=True)
    if not archive.exists():
        download(f"{BASE_URL}/{filename}", archive)
    else:
        print(f"{label}: reusing downloaded archive {archive}", flush=True)

    print(f"{label}: extracting into {MEVID_LOCAL_DIR}...", flush=True)
    extract_archive(archive, MEVID_LOCAL_DIR)
    if not expected_path.is_dir():
        raise RuntimeError(
            f"{label} extraction finished, but the required directory is missing: "
            f"{expected_path}"
        )

    marker.touch()
    archive.unlink()
    print(f"{label}: done", flush=True)

print(f"MEVID download/extraction complete: {MEVID_LOCAL_DIR}")

In [ ]:
REQUIRED_DIRS = {
    "bbox_train": MEVID_LOCAL_DIR / "bbox_train",
    "bbox_test": MEVID_LOCAL_DIR / "bbox_test",
    "annotation": MEVID_LOCAL_DIR / "mevid-v1-annotation-data",
}

for name, path in REQUIRED_DIRS.items():
    if not path.is_dir():
        raise RuntimeError(f"MEVID is incomplete. Missing directory: {name}: {path}")

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}


def count_images(directory: Path) -> int:
    count = 0
    pending = [directory]
    while pending:
        current = pending.pop()
        with os.scandir(current) as entries:
            for entry in entries:
                if entry.is_dir(follow_symlinks=False):
                    pending.append(Path(entry.path))
                elif (
                    entry.is_file(follow_symlinks=False)
                    and Path(entry.name).suffix.lower() in IMAGE_EXTENSIONS
                ):
                    count += 1
    return count


training_images = count_images(REQUIRED_DIRS["bbox_train"])
testing_images = count_images(REQUIRED_DIRS["bbox_test"])

print("====================================")
print("MEVID LOCAL DATASET READY")
print("====================================")
print("Local path:", MEVID_LOCAL_DIR)
print("Kaggle target:", KAGGLE_DATASET_HANDLE)
print("Training images:", training_images)
print("Testing images:", testing_images)

In [ ]:
if not UPLOAD_TO_KAGGLE:
    print("Kaggle upload is disabled (UPLOAD_TO_KAGGLE = False).")
else:
    upload_required_dirs = [
        MEVID_LOCAL_DIR / "bbox_train",
        MEVID_LOCAL_DIR / "bbox_test",
        MEVID_LOCAL_DIR / "mevid-v1-annotation-data",
    ]
    for path in upload_required_dirs:
        if not path.is_dir():
            raise RuntimeError(f"MEVID is incomplete. Missing directory: {path}")

    try:
        import kagglehub
    except ImportError as exc:
        raise RuntimeError(
            "KaggleHub is not installed. Install it in this local environment with "
            "`pip install -U kagglehub`, then rerun this cell."
        ) from exc

    print("Uploading MEVID to Kaggle...")
    print("If KaggleHub asks for authentication, use its normal login flow.")
    print("You can run `kagglehub.login()` in a separate cell, then rerun this cell.")

    try:
        kagglehub.dataset_upload(
            KAGGLE_DATASET_HANDLE,
            str(MEVID_LOCAL_DIR),
            version_notes="MEVID dataset uploaded from local MultiCameraTracking setup",
            ignore_patterns=[
                ".git/",
                "*/.git/",
                "__pycache__/",
                "*/__pycache__/",
                "*.pyc",
                ".DS_Store",
                "Thumbs.db",
                "*.part",
                "*.zip",
                "*.tgz",
                ".*.extracted",
            ],
        )
    except Exception as exc:
        raise RuntimeError(
            f"Failed to upload MEVID to Kaggle dataset {KAGGLE_DATASET_HANDLE}. "
            "The local files were preserved. Authenticate with KaggleHub/Kaggle "
            f"and rerun only this upload cell. Original error: {exc}"
        ) from exc

    print("====================================")
    print("KAGGLE UPLOAD COMPLETE")
    print("====================================")
    print("Dataset:")
    print("https://www.kaggle.com/datasets/qimatx/mevid")
    print("The Kaggle Notebook can now load it with:")
    print('MEVID_ROOT = Path(kagglehub.dataset_download("qimatx/mevid"))')